## Research Notebook for PISA question and answers on different languages

## Libraries

In [496]:
import os, getpass

In [497]:
from dotenv import load_dotenv, find_dotenv

In [498]:
# from openai import OpenAI
from anthropic import Anthropic

In [499]:
load_dotenv(find_dotenv(usecwd=True))  # finds .env from current working dir upward

True

In [500]:
# --- 0) Setup: imports & config
import os, re, time, json, math, random
from typing import Dict, List, Tuple, Any, Optional
import pandas as pd
import time

In [501]:
from tqdm import tqdm

## Configuration

In [502]:
client = Anthropic()

In [503]:
MODEL_CLAUDE = "claude-opus-4-5-20251101"
# claude-sonnet-4-5-20251101
# claude-sonnet-4-5-20250929

In [ ]:
SHEET_ID = "1QVPzB7uMwqJ6jCsHkwIILnXvDQIycpqkcV3bkiDpzyQ" 
WORKSHEET_NAME = "datasetM"  # change if needed "dataset"
RESULTS_CSV = "llm_eval_results_Claude.csv"
SAMPLE_PER_LANGUAGE = 5     # 5 per language
MAX_LANGUAGES = 2          # 43 languages total
SEED = 42

random.seed(SEED)

## Load data from Google Sheet

In [505]:
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={WORKSHEET_NAME}"
try:
    df = pd.read_csv(csv_url)
except Exception as e:
    raise RuntimeError(
        "Failed to read the Google Sheet via CSV export. "
        "Make sure the sheet is shared as 'Anyone with the link can view', "
        f"ID is correct, and tab name matches. Underlying error: {e}"
    )

expected_cols = {
    "qid","language","question","context","options","gold",
    "answer_type","category","difficulty","rationale","source"
}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Your sheet is missing columns: {sorted(missing)}")

In [506]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1075 entries, 0 to 1074
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   qid            1075 non-null   object
 1   language       1075 non-null   object
 2   language_code  1075 non-null   object
 3   question       1075 non-null   object
 4   context        1075 non-null   object
 5   options        1075 non-null   object
 6   gold           1075 non-null   object
 7   answer_type    1075 non-null   object
 8   category       1075 non-null   object
 9   difficulty     1075 non-null   object
 10  rationale      172 non-null    object
 11  source         1075 non-null   object
dtypes: object(12)
memory usage: 100.9+ KB


## Normalize & sample

In [507]:
df["language"] = df["language"].astype(str).str.strip()
# pick first MAX_LANGUAGES languages (sorted)
languages = sorted(df["language"].unique())[:MAX_LANGUAGES]

In [508]:
# sample up to SAMPLE_PER_LANGUAGE per language
sampled = (
    df[df["language"].isin(languages)]
    .groupby("language", sort=True, group_keys=False)
    .head(SAMPLE_PER_LANGUAGE)
    .reset_index(drop=True)
)
if sampled.empty:
    raise ValueError("No rows selected. Check your data.")

In [509]:
print(f"Selected {len(sampled)} rows across {sampled['language'].nunique()} languages.")
display(sampled[["qid","language","question","gold"]])

Selected 10 rows across 2 languages.


,qid,language,question,gold
0,q001,Albanian,Nëse Tania vendos të blejë makinën D dhe ta sh...,C
1,q002,Albanian,"Nëse trendi i shitjeve vazhdon, në cilin vit n...",C
2,q003,Albanian,Cili është numri më i madh i kutive mesatare q...,B
3,q004,Albanian,Kompania e cila jep kamionë me qera konfirmoi ...,C
4,q005,Albanian,"Mesatarisht, afërisisht sa milionë kilometra g...",D
5,q001,Arabic,إذا قررت تهاني شراء السيارة D وإعادة بيعها بع...,C
6,q002,Arabic,في حالة استمرار المبيعات في هذا الاتجاه، في أي...,C
7,q003,Arabic,ما أكبر عدد من الصناديق متوسطة الحجم يمكن أن ت...,B
8,q004,Arabic,أكدت الشركة التي تؤجّر الشاحنات أنه يمكن ملء ا...,C
9,q005,Arabic,في المتوسط، كم مليون كيلومتر تقريبًا يَبْعُد ك...,D


## Parse options

In [510]:
def parse_options(raw: str) -> Dict[str, Any]:
    """
    Parse multiple-choice options from a JSON-encoded string.

    Supported formats
    -----------------
    1) Simple labeled strings (original format):
        ["A) 1575", "B) 8925", "C) 9000", "D) 9975"]

        -> {"A": "1575", "B": "8925", "C": "9000", "D": "9975"}

    2) List of dicts with labels mapping to lists of tokens:
        [
          {"A": ["India", "Colombia"]},
          {"B": ["India", "Armenia"]},
          {"C": ["Panama", "Colombia"]},
          {"D": ["Kazakhstan", "Colombia"]}
        ]
    """
    if pd.isna(raw):
        raise ValueError("Options are empty")

    # Load JSON
    try:
        items = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format for options: {e}")

    if not isinstance(items, list):
        raise ValueError("Expected a JSON list at top level")

    # Case 1: list of strings -> original behavior
    if all(isinstance(item, str) for item in items):
        options: Dict[str, Any] = {}
        for item in items:
            # Match patterns like "A) text", "B. text", or "C: text"
            if not isinstance(item, str):
                raise ValueError(f"Option is not a string: {item}")
            match = re.match(r"^\s*([A-Z])[\)\.\:]\s*(.+)$", item.strip())
            if match:
                label, text = match.groups()
                options[label.upper()] = text.strip()
            else:
                # Fallback: assign next available letter automatically
                next_label = chr(ord('A') + len(options))
                options[next_label] = item.strip()
        return options

    # Case 2: list of dicts like [{"A": [...]}, {"B": [...]}]
    if all(isinstance(item, dict) for item in items):
        options: Dict[str, Any] = {}
        for idx, d in enumerate(items):
            if len(d) != 1:
                raise ValueError(
                    f"Each dict must have exactly one key (label). Problem at index {idx}: {d}"
                )
            (label_raw, value) = next(iter(d.items()))
            if not isinstance(label_raw, str):
                raise ValueError(f"Label must be a string, got: {label_raw}")

            label = label_raw.strip().upper()
            if not re.fullmatch(r"[A-Z]", label):
                raise ValueError(f"Invalid option label '{label_raw}' at index {idx}")

            # Accept list or scalar; normalize scalars into single-element lists if needed
            if isinstance(value, list):
                options[label] = value
            else:
                options[label] = [value]

        return options

    # If we reach here, the list is mixed or has unsupported types
    raise ValueError(
        "Unsupported options format: expected list of strings or list of single-key dicts"
    )


In [511]:
test = parse_options('["A) 2018", "B) 2019", "C) 2020", "D) 2021"]')
print(test)

{'A': '2018', 'B': '2019', 'C': '2020', 'D': '2021'}


In [512]:
options_block = "\n".join([f"{k}. {v}" for k,v in test.items()])
print(options_block)

A. 2018
B. 2019
C. 2020
D. 2021


## Build prompt

In [513]:
def build_prompt(row: pd.Series, options: Dict[str,str]) -> str:
    """
    Builds prompt for MCQ.
    """
    options_block = "\n".join([f"{k}. {v}" for k,v in options.items()])

    return (
        f"{row['context']}\n"
        f"{row['question']}\n"
        f"{options_block}\n\n"
    )

In [514]:
answer_letter_regex = re.compile(r"\s*([A-Z])\s*[.)]?\s*")

def extract_letter(text: str, valid_letters: List[str]) -> str:
    """
    Extract the first single-letter A-Z token that is in valid_letters.
    """
    if not text:
        return ""
    # First line is the letter per our format; but still be defensive:
    first_line = text.splitlines()[0].strip().rstrip(".)").upper()
    # If first line is a single valid letter, use it
    if len(first_line) == 1 and first_line.upper() in valid_letters:
        return first_line.upper()
    # Else find any A-Z token
    m = answer_letter_regex.search(text.upper())
    if m and m.group(1) in valid_letters:
        return m.group(1)
    return ""

## LLM call wrapper

In [ ]:
def llm_completion(
    prompt: str,
    model: Optional[str] = None,
    max_tokens: Optional[int] = None,
    system_prompt: str="Reply text inside the <reasoning> tags. \
        Output only the letter answer outside the tags.", #"Reply format: <LETTER>",
    retries: int = 2,
    backoff_seconds: float = 1.5,
    # reasoning_effort: Optional[str] = None,     # e.g., "low"|"medium"|"high" (and some models support "none")
    # reasoning_summary: Optional[str] = None,    # e.g., "auto" (or "concise"/"detailed" depending on model)
    **kwargs,
) -> Tuple[str, Dict[str, Any]]:
    """
    Call Anthropic Claude API and return (text, usage_dict).
    usage_dict contains: input_tokens, output_tokens, total_tokens (if present)
    """
    mdl = model or MODEL_CLAUDE
    last_err = None

    # Claude requires max_tokens
    if max_tokens is None:
        max_tokens = 4096

    effort_level = "low"

    for attempt in range(retries + 1):
        try:
            # Build arguments dynamically — include only if provided
            call_args = dict(
                model=mdl,
                betas=["effort-2025-11-24"],
                max_tokens=max_tokens,
                system=system_prompt,
                messages=[{"role": "user", "content": prompt}],
                output_config={
                    "effort": effort_level
                }
            )

            # Merge other kwargs (e.g., stop, seed, etc.)
            for k, v in kwargs.items():
                if k not in call_args:
                    call_args[k] = v

            # Actual model call
            response = client.beta.messages.create(**call_args)
            if response is None:
                return "No response from model.", {}
            
            # Extract text from response
            text = ""
            thinking_text = ""
            answer = ""
            
            for block in response.content:
                if block.type == "text":
                    text += block.text

            text = text.strip()

            def separate_thinking_and_answer(text: str) -> Tuple[str, str]:
                if not text:
                    return "", ""
                # Pattern to match <thinking>...</thinking> tags (with optional whitespace and newlines)
                thinking_pattern = r'<reasoning>(.*?)</reasoning>'
                # Extract all thinking content
                thinking_matches = re.findall(thinking_pattern, text, re.DOTALL | re.IGNORECASE)
                thinking_content = "\n\n".join(match.strip() for match in thinking_matches)
                # Remove all thinking tags and their content to get the final answer
                final_answer = re.sub(thinking_pattern, '', text, flags=re.DOTALL | re.IGNORECASE)
                final_answer = final_answer.strip()
                return thinking_content, final_answer
            thinking_text, answer = separate_thinking_and_answer(text)

            # --- Extract usage safely across SDK versions ---
            usage_obj = getattr(response, "usage", None)

            def _get(obj, key, default=None):
                if obj is None:
                    return default
                if isinstance(obj, dict):
                    return obj.get(key, default)
                return getattr(obj, key, default)

            usage: Dict[str, Any] = {
                "input_tokens": None,
                "output_tokens": None,
                "reasoning_tokens": None,
                "effort": effort_level,
                "summary": thinking_text,
            }

            if usage_obj is not None:
                # SDK objects often allow attribute access; dict-like in some contexts.
                
                usage["input_tokens"] = _get(usage_obj, "input_tokens")
                usage["output_tokens"] = _get(usage_obj, "output_tokens")

                out_details = _get(usage_obj, "output_tokens_details", {})
                usage["reasoning_tokens"] = _get(out_details, "reasoning_tokens")    
  
            return answer, thinking_text, usage

        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(backoff_seconds * (attempt + 1))
            else:
                raise

## Evaluation loop

In [516]:
def eval_rows(
        rows: pd.DataFrame, 
        cycle: int,
        model_tag: str,
        model_name: str, 
        results_path: str = RESULTS_CSV,
        sleep_s: float = 0.0, 
        retries: int = 2
    ) -> pd.DataFrame:
    results = []
    file_exists = os.path.exists(results_path)
    for i, row in tqdm(rows.iterrows(), total=len(rows), desc=f"Evaluating {model_tag} / {model_name}, cycle {cycle}"):
        qid = row["qid"]
        lang = row["language"]
        gold = str(row["gold"]).strip().upper()
        difficulty = row["difficulty"]

        # Parse options
        try:
            opts = parse_options(row["options"])
        except Exception as e:
            row_result = {
                "qid": qid,
                "language": lang,
                "pred": "",
                "gold": gold,
                "is_correct": False,
                "error": f"OptionsParseError: {e}",
                "raw": "",
                "difficulty": difficulty,
                "question": row["question"],
                "options_json": json.dumps(opts if 'opts' in locals() else {}, ensure_ascii=False),
                "model_tag": model_tag,
                "model_name": model_name,
                "cycle": cycle,
                "input_tokens": 0,
                "output_tokens": 0,
                "reasoning_tokens": 0,
                "effort": "Default",
                "summary": "",
            }
            results.append(row_result)

            # Save immediately
            pd.DataFrame([row_result]).to_csv(
                results_path,
                mode="a",
                header=not file_exists,
                index=False,
                encoding="utf-8"
            )
            file_exists = True
            continue

        valid_letters = sorted(list(opts.keys()))
        prompt = build_prompt(row, opts)

        # call model with simple retry
        raw = ""
        thinking = ""
        usage = {}
        err = ""
        for attempt in range(retries + 1):
            try:
                raw, thinking, usage = llm_completion(
                    prompt,
                    model=model_name,
                )
                break
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))
                else:
                    raw, thinking, usage = "", "", {}
        pred = extract_letter(raw, valid_letters)
        is_correct = (pred == gold)

        row_result = {
            "qid": qid,
            "language": lang,
            "pred": pred,
            "gold": gold,
            "is_correct": bool(is_correct),
            "error": err,
            "raw": raw,
            "difficulty": difficulty,
            "question": row["question"],
            "options_json": json.dumps(opts, ensure_ascii=False),
            "model_tag": model_tag,
            "model_name": model_name,
            "cycle": cycle,
            "input_tokens": usage.get("input_tokens"),
            "output_tokens": usage.get("output_tokens"),
            "reasoning_tokens": usage.get("reasoning_tokens"),
            "effort": usage.get("effort"),
            "summary": usage.get("summary", ""),
        }

        results.append(row_result)

        # *** Save this row immediately ***
        pd.DataFrame([row_result]).to_csv(
            results_path,
            mode="a",
            header=not file_exists,
            index=False,
            encoding="utf-8"
        )
        file_exists = True

        if sleep_s > 0:
            time.sleep(sleep_s)
    return pd.DataFrame(results)

## Execution

In [517]:
MODELS_TO_TEST = [
    ("Claude", MODEL_CLAUDE)
]

In [518]:
# filtered_df = df[df["language"] == "English"]
sampled = df

In [519]:
N_CYCLES = 5  # repeat the same question to the same LLM

In [520]:
if os.path.exists(RESULTS_CSV):
    existing_results = pd.read_csv(RESULTS_CSV)
    print(f"Loaded existing results from {RESULTS_CSV}: {len(existing_results)} rows")
else:
    existing_results = pd.DataFrame()
    print("No existing results file found. Starting fresh.")

for tag, model_name in MODELS_TO_TEST:
    print(f"\nEvaluating {tag} -> {model_name}")
    for cycle in range(1, N_CYCLES + 1):
        # Determine which qids are already done for this (tag, model_name, cycle)
        if existing_results.empty:
            # Nothing done yet at all
            rows_to_eval = sampled.copy()
        else:
            # Subset only rows already done for this (tag, model_name, cycle)
            subset = existing_results[
                (existing_results["model_tag"] == tag) &
                (existing_results["model_name"] == model_name) &
                (existing_results["cycle"] == cycle)
            ]

            if subset.empty:
                # No rows done yet for this model+cycle
                rows_to_eval = sampled.copy()
            else:
                # Use (qid, language) pairs as the key — more robust than qid alone
                done_pairs = set(zip(subset["qid"], subset["language"]))

                mask = ~sampled.apply(
                    lambda r: (r["qid"], r["language"]) in done_pairs,
                    axis=1
                )
                rows_to_eval = sampled[mask]

        if rows_to_eval.empty:
            print(f"  • cycle {cycle}/{N_CYCLES}: already complete, skipping")
            continue

        print(f"  • cycle {cycle}/{N_CYCLES}: evaluating {len(rows_to_eval)} questions")
        t0 = time.time()

        df_new = eval_rows(
            rows_to_eval,
            model_tag=tag,
            model_name=model_name,
            cycle=cycle,
            results_path=RESULTS_CSV
        )

        elapsed = time.time() - t0
        print(f"    Done in {elapsed:.1f}s, newly evaluated {len(df_new)} rows.")

        # Update in-memory copy so subsequent cycles/ models can see freshly written rows
        existing_results = pd.concat([existing_results, df_new], ignore_index=True)

No existing results file found. Starting fresh.

Evaluating Claude -> claude-opus-4-5-20251101
  • cycle 1/5: evaluating 1075 questions


Evaluating Claude / claude-opus-4-5-20251101, cycle 1: 100%|██████████| 1075/1075 [2:22:27<00:00,  7.95s/it] 


    Done in 8547.3s, newly evaluated 1075 rows.
  • cycle 2/5: evaluating 1075 questions


Evaluating Claude / claude-opus-4-5-20251101, cycle 2: 100%|██████████| 1075/1075 [2:18:30<00:00,  7.73s/it] 


    Done in 8310.2s, newly evaluated 1075 rows.
  • cycle 3/5: evaluating 1075 questions


...

    Done in 8435.9s, newly evaluated 1075 rows.
  • cycle 4/5: evaluating 1075 questions


Evaluating Claude / claude-opus-4-5-20251101, cycle 4: 100%|██████████| 1075/1075 [2:25:00<00:00,  8.09s/it] 


    Done in 8700.3s, newly evaluated 1075 rows.
  • cycle 5/5: evaluating 1075 questions


Evaluating Claude / claude-opus-4-5-20251101, cycle 5: 100%|██████████| 1075/1075 [2:19:26<00:00,  7.78s/it] 


    Done in 8366.6s, newly evaluated 1075 rows.


In [521]:
res_df = pd.read_csv(RESULTS_CSV)
print(f"\nTotal results loaded: {len(res_df)}")


Total results loaded: 5375


## Results

In [522]:
overall_by_model = (
    res_df.groupby(["model_tag","model_name"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by model:")
display(overall_by_model)


Overall accuracy by model:


,model_tag,model_name,accuracy
0,Claude,claude-opus-4-5-20251101,0.959628


In [523]:
overall_by_question = (
    res_df.groupby(["qid"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by question:")
display(overall_by_question)


Overall accuracy by question:


,qid,accuracy
0,q001,1.000000
2,q003,1.000000
4,q005,1.000000
6,q007,1.000000
5,q006,1.000000
8,q009,1.000000
16,q017,1.000000
14,q015,1.000000
12,q013,1.000000
24,q025,1.000000


In [524]:
overall_by_lang = (
    res_df.groupby(["language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by lang:")
display(overall_by_lang)


Overall accuracy by lang:


,language,accuracy
18,Georgian,1.000
15,Finnish,1.000
13,English,1.000
21,Hebrew,1.000
34,Portuguese,1.000
40,Swedish,1.000
42,Turkish,1.000
26,Japanese,0.992
5,Bosnian,0.992
7,Catalan,0.992


In [525]:
by_question = (
    res_df.groupby(["model_tag","model_name","language","qid"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["accuracy"])
)

# print("\nAccuracy by question:")
# display(by_question)

accuracy_counts_total = (
    by_question["accuracy"]
    .value_counts()
    .rename_axis("accuracy")
    .reset_index(name="count")
    .sort_values("accuracy")
)

print("\nCount of total questions by accuracy:")
display(accuracy_counts_total)

accuracy_counts = (
    by_question
    .groupby(["model_tag", "model_name", "accuracy"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["model_tag", "model_name", "accuracy"])
)

print("\nCount of questions by accuracy:")
display(accuracy_counts)


Count of total questions by accuracy:


,accuracy,count
1,0.0,27
4,0.2,8
5,0.4,6
3,0.6,9
2,0.8,14
0,1.0,1011



Count of questions by accuracy:


,model_tag,model_name,accuracy,count
0,Claude,claude-opus-4-5-20251101,0.0,27
1,Claude,claude-opus-4-5-20251101,0.2,8
2,Claude,claude-opus-4-5-20251101,0.4,6
3,Claude,claude-opus-4-5-20251101,0.6,9
4,Claude,claude-opus-4-5-20251101,0.8,14
5,Claude,claude-opus-4-5-20251101,1.0,1011


In [526]:
def safe_acc(s):
    return float('nan') if s.empty else s.mean()
overall_acc = safe_acc(res_df["is_correct"])
print(f"\nCombined overall accuracy on {len(res_df)} items: {overall_acc:.3f}")


Combined overall accuracy on 5375 items: 0.960


In [527]:
for tag, _ in MODELS_TO_TEST:
    out_path = f"llm_eval_results__{tag}.csv"
    res_df.query("model_tag == @tag").to_csv(out_path, index=False)
    print(f"Saved {tag} results to: {out_path}")

# (Optional) quick peek
display(res_df.head())

Saved Claude results to: llm_eval_results__Claude.csv


,qid,language,pred,gold,is_correct,error,raw,difficulty,question,options_json,model_tag,model_name,cycle,input_tokens,output_tokens,reasoning_tokens,effort,summary
0,q001,Albanian,C,C,True,NaN,C,level 6,Nëse Tania vendos të blejë makinën D dhe ta sh...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",Claude,claude-opus-4-5-20251101,1,236,251,NaN,high,"Për të gjetur vlerën e makinës pas tre vitesh,..."
1,q002,Albanian,C,C,True,NaN,C. 2020,level 6,"Nëse trendi i shitjeve vazhdon, në cilin vit n...","{""A"": ""2015"", ""B"": ""2018"", ""C"": ""2020"", ""D"": ""...",Claude,claude-opus-4-5-20251101,1,149,364,NaN,high,Duhet të gjej vitin kur numri i DVD-ve të shit...
2,q003,Albanian,B,B,True,NaN,B. 128,level 2,Cili është numri më i madh i kutive mesatare q...,"{""A"": ""320"", ""B"": ""128"", ""C"": ""26"", ""D"": ""16""}",Claude,claude-opus-4-5-20251101,1,276,304,NaN,high,Për të gjetur numrin më të madh të kutive mesa...
3,q004,Albanian,C,C,True,NaN,C,level 6,Kompania e cila jep kamionë me qera konfirmoi ...,"{""A"": ""Ajo ka të drejtë, sepse lartësia e një ...",Claude,claude-opus-4-5-20251101,1,651,726,NaN,high,Let me analyze this problem step by step.\n\nF...
4,q005,Albanian,D,D,True,NaN,D,level 2,"Mesatarisht, afërisisht sa milionë kilometra g...","{""A"": ""5 milionë km"", ""B"": ""30 milionë km"", ""C...",Claude,claude-opus-4-5-20251101,1,217,144,NaN,high,Për të gjetur distancën mesatare nga Dielli te...


In [528]:
df_false = res_df[res_df['is_correct'] == False]
display(df_false.head(50))

,qid,language,pred,gold,is_correct,error,raw,difficulty,question,options_json,model_tag,model_name,cycle,input_tokens,output_tokens,reasoning_tokens,effort,summary
10,q011,Albanian,A,B,False,NaN,A,level 6,Helena mendon se Koreja e Jugut ka më tepër si...,"{""A"": ""Po"", ""B"": ""Jo""}",Claude,claude-opus-4-5-20251101,1,627,349,NaN,high,Të analizoj të dhënat për të parë nëse Koreja ...
34,q010,Arabic,C,A,False,NaN,"C. ['بنما ', 'كولومبيا ']",level 6,بالنظر في الفترتين الزمنيتين: 2005 إلى 2010...,"{""A"": [""الهند"", ""كولومبيا ""], ""B"": [""الهند"", ""...",Claude,claude-opus-4-5-20251101,1,790,1009,NaN,high,لحساب التغيير الأكبر في النسبة المئوية، سأحسب ...
60,q011,Azerbaijani / Azeri,A,B,False,NaN,A,level 6,"Həlimə iddia edir ki, göstərilmiş illər üçün d...","{""A"": ""Bəli"", ""B"": ""Xeyr""}",Claude,claude-opus-4-5-20251101,1,676,425,NaN,high,"Həlimənin iddiasını yoxlayaq: ""Ölkə M-də daha ..."
85,q011,Basque,A,B,False,NaN,A,level 6,Haizeak baieztatu du Hego Koreak zerrendako be...,"{""A"": ""Bai"", ""B"": ""Ez""}",Claude,claude-opus-4-5-20251101,1,615,307,NaN,high,Haizeak baieztatu du Hego Koreak zerrendako be...
117,q018,Bokmål,C,A,False,NaN,C,level 5,Er påstanden fakta eller mening?\nNye studier ...,"{""A"": [""Mening"", ""Fakta"", ""Fakta"", ""Mening""], ...",Claude,claude-opus-4-5-20251101,1,1515,377,NaN,high,Let me analyze each statement to determine if ...
159,q010,Bulgarian,C,A,False,NaN,C,level 6,Разгледайте двата времеви периода: от 2005 до ...,"{""A"": [""Казахстан"", ""Колумбия""], ""B"": [""Казахс...",Claude,claude-opus-4-5-20251101,1,667,727,NaN,high,Трябва да намеря държавите с най-значителни из...
210,q011,Chinese,A,B,False,NaN,A,level 6,韩琳娜指出，在所列年份中，韩国比列表中任何其他国家拥有更大的森林面积。\n她的说法是否与电子...,"{""A"": ""是"", ""B"": ""否""}",Claude,claude-opus-4-5-20251101,1,617,272,NaN,high,"让我分析韩琳娜的说法。\n\n她说：""在所列年份中，韩国比列表中任何其他国家拥有更大的森林面..."
215,q016,Chinese,NaN,B,False,NaN,"根据文中IDFA的陈述：""牛奶含有九种基本营养素，是完整的营养组合。牛奶除了是钙和维生素D的...",level 1b,根据IDFA的说法，下列哪一项陈述是很多重要的保健专业人士和团体同意的？,"{""A"": ""饮用牛奶和牛奶制品会导致肥胖。"", ""B"": ""牛奶是基本维生素和矿物质的良好...",Claude,claude-opus-4-5-20251101,1,730,769,NaN,high,让我分析这道阅读理解题。\n\n题目问的是：根据IDFA（国际乳制品产业协会）的说法，哪一项...
260,q011,Czech,A,B,False,NaN,A,level 6,"Helena tvrdí, že v uvedených letech má Jižní K...","{""A"": ""Ano"", ""B"": ""Ne""}",Claude,claude-opus-4-5-20251101,1,590,368,NaN,high,Podívám se na data pro Jižní Koreu a porovnám ...
284,q010,Danish,B,A,False,NaN,"B. ['Indien', 'Armenien']",level 6,Undersøg følgende to perioder: fra 2005 til 20...,"{""A"": [""Indien"", ""Colombia""], ""B"": [""Indien"", ...",Claude,claude-opus-4-5-20251101,1,611,999,NaN,high,Jeg skal finde de to lande med den største sti...


In [529]:
df_summary = (
    df_false
    .groupby(['qid'])
    .size()
    .reset_index(name='num_incorrect')
)

print(df_summary)


     qid  num_incorrect
0   q002              2
1   q004              6
2   q008              1
3   q010             51
4   q011             96
5   q012              5
6   q014              5
7   q016              1
8   q018             25
9   q019              8
10  q022              1
11  q023             16
